# Spacecraft Collision Avoidance

## 1. Imports

In [ ]:
import json
import pickle
import re
import numpy as np
import pandas as pd
import lightgbm as lgb
import matplotlib.pyplot as plt
import plotly.express as px
import os
from ydata_profiling import ProfileReport
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

%matplotlib inline


## 2. Load data

In [ ]:
df_train_raw = pd.read_csv('train_data.csv')
df_test_raw = pd.read_csv('test_data.csv')

#shape of the dataframes and number of unique events in each
print("Train DataFrame shape:", df_train_raw.shape, "| unique events:", df_train_raw['event_id'].nunique())
print("Test DataFrame shape:", df_test_raw.shape, "| unique events:", df_test_raw['event_id'].nunique())


In [ ]:
#train and test time to close approach min and max
print(df_train_raw["time_to_tca"].describe().loc[["min","max"]])
print(df_test_raw["time_to_tca"].describe().loc[["min","max"]])

In [ ]:
# Precompute grouped counts once
event_counts = df_train_raw.groupby("event_id")["time_to_tca"]

# Count events with at least one CDM < 2 days
events_lt_2 = (event_counts.min() < 2).sum()

# Count events with at least one CDM > 2 days
events_gt_2 = (event_counts.max() > 2).sum()

# Total number of events
total_events = event_counts.size()

print("number of events that have < 2.0 days data :", events_lt_2)
print("number of events that have > 2.0 days data :", events_gt_2)
print("total number of events :",len(total_events))

# Lists of event_ids
events_less_than_two_days = event_counts.min()[event_counts.min() < 2].index.tolist()
events_more_than_two_days = event_counts.max()[event_counts.max() > 2].index.tolist()


In [ ]:
# Identify events that are present in both categories
events_trainable = [value for value in events_less_than_two_days if value in events_more_than_two_days]
print("number of trainable events : ",len(events_trainable))

## 3. Build the dataset

For each event, per the challenge's three eligibility rules:

- **(i)** Only CDMs with `time_to_tca >= 2` may be used as inputs - this is the actual decision point (avoidance manoeuvres are planned at least 2 days out). We take the **most recent** such CDM per event as the feature row.
- **(ii)** The event's true final CDM (`time_to_tca` minimum overall) must be within 1 day of TCA - its `risk` is the prediction **target**, renamed `final_risk` so it's never confused with a row's own self-reported `risk`.
- **(iii)** The event needs at least one eligible (`>=2` days out) CDM to learn from, in addition to that final CDM as the target - i.e. **>= 2 CDMs total**, not >= 2 eligible CDMs.

 Overall, three constraints were put on events to be eligible in the test set

In [ ]:
def build_event_level_dataset(df_raw, require_target=True):
    """Collapse a raw CDM-level dataframe into one row per eligible event.

    Parameters
    ----------
    df_raw : the raw per-CDM dataframe (e.g. df_train_raw or df_test_raw).
    require_target : True for files that still contain each event's true final CDM
        (e.g. the unfiltered train_data.csv), so a `final_risk` label can be derived
        locally. False for files that have had the near-TCA CDMs stripped out (e.g. the
        released test_data.csv) -- in that case you get features only; the ground truth
        for those events isn't present in the file and must come from elsewhere.
    """
    # Constraint (iii): only CDMs >= 2 days before TCA can be used as inputs
    eligible = df_raw[df_raw['time_to_tca'] >= 2]

    # Most recent eligible CDM per event = decision-point feature row
    latest_eligible_idx = eligible.groupby('event_id')['time_to_tca'].idxmin()
    latest_eligible = df_raw.loc[latest_eligible_idx].reset_index(drop=True)

    if not require_target:
        # e.g. test_data.csv: no CDM with time_to_tca < 2 exists in the file at all,
        # so there is nothing to derive a final_risk label from -- features only.
        return latest_eligible

    # Final CDM per event = the row with the smallest time_to_tca overall
    final_idx = df_raw.groupby('event_id')['time_to_tca'].idxmin()
    final_cdm = df_raw.loc[final_idx, ['event_id', 'time_to_tca', 'risk']].rename(
        columns={'time_to_tca': 'final_time_to_tca', 'risk': 'final_risk'}
    )

    events = final_cdm.set_index('event_id').join(
        latest_eligible.set_index('event_id'), how='inner'  # constraint (i): needs an eligible CDM too
    )

    # Constraint (ii): true final CDM must be within 1 day of TCA
    events = events[events['final_time_to_tca'] < 1]

    return events.reset_index()


challenge_train = build_event_level_dataset(df_train_raw, require_target=True)
challenge_test = build_event_level_dataset(df_test_raw, require_target=False)



print("Eligible train events:", len(challenge_train), "/", df_train_raw['event_id'].nunique())
print("Test events (features only, no local ground truth):", len(challenge_test),
      "/", df_test_raw['event_id'].nunique())

# Sanity checks
assert challenge_train['event_id'].is_unique
assert (challenge_train['time_to_tca'] >= 2).all()
assert (challenge_train['final_time_to_tca'] < 1).all()
assert challenge_test['event_id'].is_unique
assert (challenge_test['time_to_tca'] >= 2).all()
challenge_train.head()


In [ ]:
print("Train DataFrame shape:", challenge_train.shape, "| unique events:", challenge_train['event_id'].nunique())

In [ ]:
# Analyse risk distribution
high_risk_count = (challenge_train['final_risk'] >= -6.0).sum()
low_risk_count = (challenge_train['final_risk'] < -6.0).sum()
print("High risk:", high_risk_count)
print("Low risk:", low_risk_count)
print("Ratio:", (challenge_train['final_risk'] >= -6.0).sum() /(low_risk_count) * 100)

In [ ]:
# Uses the latest available risk value to determine the normal and anomalous cases
challenge_train['predicted_risk'] = 0 
challenge_train.loc[challenge_train['risk']<-6.0,'predicted_risk'] =  -6.000000000000001
challenge_train.loc[challenge_train['risk']>=-6.0,'predicted_risk'] =  challenge_train.loc[challenge_train['risk'] >=-6.0].risk.values

In [ ]:
# Assigns the classes based on risk values
challenge_train['predicted_risk_class']=np.nan
challenge_train.loc[challenge_train['predicted_risk']>=-6.0,'predicted_risk_class'] = int(1)
challenge_train.loc[challenge_train['predicted_risk']<-6.0,'predicted_risk_class'] = int(0)
print("number of high risk events with above criterion",challenge_train[challenge_train['predicted_risk_class']==1].shape[0])
print("number of low risk events with above criterion",challenge_train[challenge_train['predicted_risk_class']==0].shape[0])
print("ratio of high risk versus low risk with above criterion",challenge_train[challenge_train['predicted_risk_class']==1].shape[0]/challenge_train[challenge_train['predicted_risk_class']==0].shape[0]*100)

In [ ]:
challenge_train['predicted_risk_anomaly']=np.nan

challenge_train.loc[(challenge_train['predicted_risk']>=-6.0)&(challenge_train['final_risk']>=-6.0),'predicted_risk_anomaly'] = int(0) # both predicted and final risk are high risk
challenge_train.loc[(challenge_train['predicted_risk']<-6.0)&(challenge_train['final_risk']<-6.0),'predicted_risk_anomaly'] = int(1) # both predicted and final risk are low risk
challenge_train.loc[(challenge_train['predicted_risk']>=-6.0)&(challenge_train['final_risk']<-6.0),'predicted_risk_anomaly'] = int(2) # predicted high, final low
challenge_train.loc[(challenge_train['predicted_risk']<-6.0)&(challenge_train['final_risk']>=-6.0),'predicted_risk_anomaly'] = int(3) # predicted low, final high

print("number of 0 to 1 risk anomaly events with above criterion",challenge_train[challenge_train['predicted_risk_anomaly']==3].shape[0])
print("number of 1 to 0 risk anomaly events with above criterion",challenge_train[challenge_train['predicted_risk_anomaly']==2].shape[0])
print("number of 0 to 0 risk events with above criterion",challenge_train[challenge_train['predicted_risk_anomaly']==1].shape[0])
print("number of 1 to 1 risk events with above criterion",challenge_train[challenge_train['predicted_risk_anomaly']==0].shape[0])

## 5. Missing values

In [ ]:
# Calculate the percentage of missing values for each column
null_percentage = challenge_train.isnull().mean() * 100
complete_rows = challenge_train.dropna().shape[0]

print('Events with no missing values:', complete_rows, '/', challenge_train.shape[0])
print('Fraction of events with at least one missing value: '
      f"{1 - complete_rows / challenge_train.shape[0]:.2%}")
print('\nColumns with the most missing values:')
null_percentage.sort_values(ascending=False).head(10)

## 6. (Optional) Full profiling report

Skip this cell if you just want the model — it's slow on large frames and is exploratory only.

In [ ]:
profile = ProfileReport(challenge_train, minimal=True, explorative=False, title="Challenge Train Data Profile")
profile.to_file("train_data_profile.html")

## 7. Feature selection

Numeric features from the decision-point CDM. `event_id` is excluded ( not a predictive feature). `risk` (the CDM's own self-reported risk at the decision point) is included as a feature - it's legitimately available at decision time, unlike `final_risk`.

In [ ]:
# load or create your dataset
df_train = challenge_train.copy()

# Prepare features and target
X = df_train.drop(columns=["final_risk", "event_id", "mission_id", "final_time_to_tca"])
y = df_train["final_risk"]
X = pd.get_dummies(X, drop_first=True)
w = (y > 0.001).astype(int) * 10 + 1
X_train, X_val, y_train, y_val, w_train, w_val = train_test_split(X, y, w, test_size=0.2, random_state=42)

# Create LightGBM datasets
train_data = lgb.Dataset(X_train, y_train, weight=w_train)
valid_data = lgb.Dataset(X_val, y_val, weight=w_val)

# specify the configurations
params = {
    "objective": "regression",
    "metric": "rmse",
    "num_leaves": 40,
    "learning_rate": 0.05,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "verbose": -1,
}
df_train.dtypes[df_train.dtypes == "object"]


model = lgb.train(
    params, 
    train_data, 
    num_boost_round=1000, 
    valid_sets=[train_data],
    valid_names=["validation"], 
    callbacks=[lgb.early_stopping(stopping_rounds=50), lgb.log_evaluation(period=50)]
)


# save model to file
model.save_model("model1.txt")

gain_importance = model.feature_importance(importance_type="gain")
split_importance = model.feature_importance(importance_type="split")
#print("Feature importance (gain): ", gain_importance)
#print("Feature importance (split): ", split_importance)

In [16]:
# manually selected top features based on correlation and feature importance
top_features = ["relative_position_r",
    "relative_velocity_t",
    "mahalanobis_distance",
    "c_position_covariance_det",
    "c_sigma_t",
    "t_span",
    "t_rcs_estimate",
    "miss_distance",
    "risk",
    "max_risk_estimate",
    "max_risk_scaling"
]

## 8. Train / validation split


In [ ]:
X = challenge_train[top_features]
TARGET = "final_risk"
y = challenge_train[TARGET]
RISK_THRESHOLD = -6
high_risk_label = (y >= RISK_THRESHOLD).astype(int)  # Create a binary label for high-risk events
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=high_risk_label
)

# Upweight higher-risk events so the rare dangerous ones aren't drowned out by the many low-risk ones

HIGH_RISK_WEIGHT = 190

w_train = (y_train >= RISK_THRESHOLD).astype(int) * HIGH_RISK_WEIGHT + 1
w_valid = (y_valid >= RISK_THRESHOLD).astype(int) * HIGH_RISK_WEIGHT + 1

train_data = lgb.Dataset(X_train, label=y_train, weight=w_train, free_raw_data=False)
valid_data = lgb.Dataset(X_valid, label=y_valid, weight=w_valid, reference=train_data, free_raw_data=False)


In [ ]:

train_high_risk = (y_train >= RISK_THRESHOLD).mean()
valid_high_risk = (y_valid >= RISK_THRESHOLD).mean()

print(f"Train size: {len(y_train)}")
print(f"Validation size: {len(y_valid)}")
print(f"Train high-risk proportion: {train_high_risk:.3%}")
print(f"Validation high-risk proportion: {valid_high_risk:.3%}")

## 9. Train the model

In [ ]:
params = {
    "objective": "regression",
    "metric": "rmse",
    "num_leaves":63,
    "learning_rate": 0.05,
    "feature_fraction": 1.0,  # already hand-selected the features
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "verbose": -1,
}

model = lgb.train(
    params,
    train_data,
    num_boost_round=500,
    valid_sets=[train_data, valid_data],
    valid_names=["train", "valid"],
    callbacks=[lgb.early_stopping(stopping_rounds=50)],
)

print("Finished training. Best iteration:", model.best_iteration)


In [ ]:
# Evaluate on validation set
y_valid_pred = model.predict(X_valid, num_iteration=model.best_iteration)
baseline_risk_valid = challenge_train.loc[X_valid.index, "risk"].values
y_valid_pred_blended = np.maximum(y_valid_pred, baseline_risk_valid)
weighted_valid_rmse = np.sqrt(np.average((y_valid - y_valid_pred_blended) ** 2,weights=w_valid))
unweighted_valid_rmse = np.sqrt(mean_squared_error(y_valid, y_valid_pred_blended))

print("Weighted validation RMSE:", weighted_valid_rmse)
print("Unweighted validation RMSE:", unweighted_valid_rmse)


## 10. Save the model

In [ ]:
model.save_model("model.txt")

with open("model.json", "w") as f:
    json.dump(model.dump_model(), f, indent=4)

with open("model.pkl", "wb") as f:
    pickle.dump(model, f)

print("Model saved")


## 11. Generate predictions on the test set

In [ ]:
# Generate predictions for the test set
X_test = challenge_test[top_features]
y_test_pred = model.predict(X_test, num_iteration=model.best_iteration)
y_test_pred_blended = np.maximum(y_test_pred, challenge_test["risk"].values)
# Save predictions to CSV
test_predictions = challenge_test[['event_id']].copy()
test_predictions['final_risk_predicted'] = y_test_pred_blended
test_predictions.to_csv("test_predictions.csv", index=False)
print("Saved predictions to test_predictions.csv")
test_predictions.sort_values("final_risk_predicted", ascending=False).head(20)


In [ ]:
# Diagnostic plots against the VALIDATION split (the only place we have true final_risk
plt.scatter(y_valid, y_valid_pred, alpha=0.5)
plt.xlabel("Actual final risk")
plt.ylabel("Predicted final risk")
plt.title("Predicted vs Actual Final Risk (validation events)")
plt.show()

# Diagnostic plots against the TEST split (the only place we have true final_risk
plt.scatter(y_test_pred, challenge_test["risk"], alpha=0.5)
plt.xlabel("Predicted final risk")
plt.ylabel("Baseline risk (latest available)")
plt.title("Predicted vs Baseline Risk (test events)")
plt.show()
residuals = y_valid - y_valid_pred
plt.hist(residuals, bins=50)
plt.title("Residual Distribution (validation events)")
plt.show()

# Distribution of predicted risk on the actual test events
plt.hist(y_test_pred, bins=50)
plt.title("Predicted Final Risk Distribution (test events)")
plt.show()


In [ ]:

#  LightGBM predictions
y_valid_pred_lgbm = model.predict(
    X_valid,
    num_iteration=model.best_iteration
)

#  ESA LRP baseline
# Latest available risk is the prediction
baseline_risk = challenge_train.loc[X_valid.index, "risk"].values

y_valid_pred_baseline = np.where(
    baseline_risk >= -6.0,
    baseline_risk,
    -6.0
)

#  Overall RMSE
baseline_rmse = np.sqrt(mean_squared_error(y_valid, y_valid_pred_baseline))
lgbm_rmse = np.sqrt(mean_squared_error(y_valid, y_valid_pred_lgbm))

print("RMSE comparison")
print("----------------------------")
print(f"ESA baseline RMSE: {baseline_rmse:.6f}")
print(f"LightGBM RMSE:     {lgbm_rmse:.6f}")


# Define high risk events

HIGH_RISK_THRESHOLD = -6.0

actual_high_risk = y_valid >= HIGH_RISK_THRESHOLD

print("\nHigh-risk validation events:", actual_high_risk.sum())
print("Total validation events:", len(y_valid))
print("High-risk proportion:", f"{actual_high_risk.mean():.2%}")

